<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/13_trajectory_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trajectory Optimization with SQP

In this exercise, you will implement and experiment with **Sequential Quadratic Programming (SQP)** for nonlinear trajectory optimization.



In this notebook, you will experiment with **Sequential Quadratic Programming (SQP)** for nonlinear (open-loop) trajectory optimization. We consider a simplified rocket based on inverted cart-pole dynamics with thrust vectoring.

We model a rocket as an **inverted pendulum** mounted on a translating base. The system consists of:
- A thruster/base of mass $m_t$ (the "cart")
- A rocket body of mass $m_b$ and length $\ell$ (the "pole")
- The rocket can translate in 2D and rotate about its base

**State**: $x = [p_x, p_z, \theta, v_x, v_z, \omega]^\top$ where:
- $(p_x, p_z)$ = position of rocket base
- $\theta$ = angle from vertical (upright is $\theta = 0$), CW is positive.
- $(v_x, v_z)$ = base velocities  
- $\omega = \dot{\theta}$ = angular velocity of the rocket body.

**Control**: $u = [T_x, T_z]^\top$ = thrust forces (horizontal and vertical)

**Goal**: starting from a high position with initial velocity, **land the rocket softly** on a circular landing pad with:
- Small final velocity
- Upright orientation ($\theta \approx 0$)
- Position on the landing pad

**Constraints**:
1. **Thrust limits**: Upper and lower bounds on horizontal and vertical thrust forces
2. **Landing zone**: Must land within the target circular region
3. **Obstacle avoidance**: Circular no-fly zones to avoid during flight


In [ ]:
!pip install equinox

In [ ]:
# imports
import cvxpy as cp
import ipywidgets as widgets
import jax
import jax.numpy as jnp
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import HBox, IntSlider, Play, VBox, interact, interactive_output, jslink
import warnings
import functools
import equinox as eqx
from typing import Callable
from matplotlib.patches import Circle


warnings.filterwarnings("ignore", message="Objective contains too many subexpressions")

In [ ]:
# @title Helper functions


def draw_rocket(ax, p_x, p_z, theta, body_width=0.25, body_height=0.6, zorder=10):
    """
    Draw a rocket at given position and orientation.

    Args:
        ax: matplotlib axis
        p_x, p_z: position coordinates
        theta: angle from vertical (radians)
        body_width: width of rocket body
        body_height: height of rocket body
        zorder: drawing layer
    """
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    # Rocket body (rectangle)
    body_local = np.array(
        [
            [-body_width / 2, 0],
            [body_width / 2, 0],
            [body_width / 2, body_height],
            [-body_width / 2, body_height],
        ]
    )

    rotation_matrix = np.array([[cos_theta, -sin_theta], [sin_theta, cos_theta]])
    body_rotated = body_local @ rotation_matrix.T
    body_global = body_rotated + np.array([p_x, p_z])

    rocket_body = plt.Polygon(
        body_global,
        facecolor="darkgray",
        edgecolor="black",
        linewidth=2,
        alpha=0.9,
        zorder=zorder,
    )
    ax.add_patch(rocket_body)

    # Nose cone (triangle)
    nose_height = 0.15
    nose_local = np.array(
        [
            [-body_width / 2, body_height],
            [body_width / 2, body_height],
            [0, body_height + nose_height],
        ]
    )
    nose_rotated = nose_local @ rotation_matrix.T
    nose_global = nose_rotated + np.array([p_x, p_z])

    rocket_nose = plt.Polygon(
        nose_global,
        facecolor="darkgray",
        edgecolor="black",
        linewidth=2,
        alpha=0.9,
        zorder=zorder,
    )
    ax.add_patch(rocket_nose)


def draw_velocity_fire(ax, p_x, p_z, theta, v_x, v_z, flame_width=0.08, zorder=8):
    """
    Draw velocity-based fire emanating from rocket base.

    Args:
        ax: matplotlib axis
        p_x, p_z: rocket position
        theta: rocket angle (radians)
        v_x, v_z: velocity components
        flame_width: width of flame rectangle
        zorder: drawing layer (should be < rocket zorder)
    """
    vel_mag = np.sqrt(v_x**2 + v_z**2)

    if vel_mag > 0.05:  # Only draw if moving
        # Flame direction follows velocity
        flame_dir_x = v_x / vel_mag
        flame_dir_y = v_z / vel_mag

        # Flame length proportional to velocity
        flame_length = vel_mag * 0.4
        flame_length = min(flame_length, 1.5)  # Cap maximum

        # Rocket orientation
        cos_theta = np.cos(theta)
        sin_theta = np.sin(theta)

        # Flame emanates from rocket base
        rotation_matrix = np.array([[cos_theta, -sin_theta], [sin_theta, cos_theta]])
        flame_base_local = np.array([0, 0])
        flame_base_global = flame_base_local @ rotation_matrix.T + np.array([p_x, p_z])

        # Flame end point
        flame_end_x = flame_base_global[0] + flame_dir_x * flame_length
        flame_end_y = flame_base_global[1] + flame_dir_y * flame_length

        # Perpendicular vector for width
        perp_x = -flame_dir_y
        perp_y = flame_dir_x

        # Flame rectangle corners
        flame_corners = np.array(
            [
                [
                    flame_base_global[0] + perp_x * flame_width / 2,
                    flame_base_global[1] + perp_y * flame_width / 2,
                ],
                [
                    flame_base_global[0] - perp_x * flame_width / 2,
                    flame_base_global[1] - perp_y * flame_width / 2,
                ],
                [
                    flame_end_x - perp_x * flame_width / 2,
                    flame_end_y - perp_y * flame_width / 2,
                ],
                [
                    flame_end_x + perp_x * flame_width / 2,
                    flame_end_y + perp_y * flame_width / 2,
                ],
            ]
        )

        # Draw flame
        flame = plt.Polygon(
            flame_corners,
            facecolor="red",
            edgecolor="darkred",
            linewidth=1,
            alpha=0.8,
            zorder=zorder,
        )
        ax.add_patch(flame)


def draw_environment(
    ax,
    landing_pad_radius,
    x_range=4,
    water_level=0,
    obstacle1=None,
    obstacle2=None,
    rocket_radius=0.0,
):
    """
    Draw the environment: obstacles, landing pad, and water.

    Args:
        ax: matplotlib axis
        obstacle1, obstacle2: dicts with 'center' and 'radius'
        rocket_radius: safety buffer
        landing_pad_radius: radius of landing zone
        x_range: horizontal extent
        water_level: height of water surface
    """
    # Obstacles (solid and inflated safety zone)
    if obstacle1 is not None:
        obs1_circle = patches.Circle(
            obstacle1["center"],
            obstacle1["radius"],
            color="red",
            alpha=0.3,
            label="Obstacles",
            zorder=3,
        )
        obs1_inflated = patches.Circle(
            obstacle1["center"],
            obstacle1["radius"] + rocket_radius,
            color="red",
            alpha=0.1,
            linestyle="--",
            zorder=2,
        )
        ax.add_patch(obs1_circle)
        ax.add_patch(obs1_inflated)

    if obstacle2 is not None:
        obs2_circle = patches.Circle(
            obstacle2["center"], obstacle2["radius"], color="red", alpha=0.3, zorder=3
        )
        obs2_inflated = patches.Circle(
            obstacle2["center"],
            obstacle2["radius"] + rocket_radius,
            color="red",
            alpha=0.1,
            linestyle="--",
            zorder=2,
        )
        ax.add_patch(obs2_circle)
        ax.add_patch(obs2_inflated)

    # Landing pad
    landing_pad = plt.Circle(
        (0, 0),
        landing_pad_radius,
        color="green",
        alpha=0.4,
        label="Landing pad",
        zorder=5,
    )
    ax.add_patch(landing_pad)

    # Water
    ax.fill_between(
        [-x_range, x_range],
        water_level - 1.0,
        water_level,
        color="lightblue",
        alpha=0.5,
        label="Water",
        zorder=1,
    )
    ax.axhline(water_level, color="deepskyblue", linewidth=2, alpha=0.7, zorder=1)


def plot_trajectory_analysis(
    X_traj,
    U_traj,
    T_sim,
    initial_state,
    obstacle1=None,
    obstacle2=None,
    rocket_radius=0.0,
    landing_pad_radius=0.5,
    hover_thrust=None,
    u_min=None,
    u_max=None,
    title_prefix="Trajectory",
):
    """
    Plot comprehensive trajectory analysis with 6 subplots:
    - States over time (position, angle, velocity)
    - Control inputs over time
    - 2D trajectory with environment

    Args:
        X_traj: (N+1, 6) state trajectory
        U_traj: (N, 2) control trajectory
        T_sim: simulation time
        initial_state: (6,) initial state
        obstacle1, obstacle2: optional obstacle dicts
        rocket_radius: safety buffer
        landing_pad_radius: landing zone radius
        hover_thrust: hover thrust value for reference
        u_min, u_max: control bounds
        title_prefix: prefix for figure title
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f"{title_prefix} Analysis", fontsize=16, fontweight="bold")

    t_plot = np.linspace(0, T_sim, len(X_traj))
    t_control = np.linspace(0, T_sim, len(U_traj))

    # --- Subplot 1: Horizontal Position ---
    axes[0, 0].plot(t_plot, X_traj[:, 0], "b-", linewidth=2, label="$p_x$")
    axes[0, 0].axhline(0, color="g", linestyle="--", alpha=0.5, label="Target")
    axes[0, 0].set_ylabel("$p_x$ (m)", fontsize=11)
    axes[0, 0].set_xlabel("Time (s)", fontsize=11)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    axes[0, 0].set_title("Horizontal Position")

    # --- Subplot 2: Vertical Position ---
    axes[0, 1].plot(t_plot, X_traj[:, 1], "b-", linewidth=2, label="$p_z$")
    axes[0, 1].axhline(
        0, color="deepskyblue", linestyle="--", alpha=0.5, label="Water surface"
    )
    axes[0, 1].set_ylabel("$p_z$ (m)", fontsize=11)
    axes[0, 1].set_xlabel("Time (s)", fontsize=11)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    axes[0, 1].set_title("Vertical Position")

    # --- Subplot 3: Angle ---
    axes[0, 2].plot(
        t_plot, X_traj[:, 2] * 180 / np.pi, "b-", linewidth=2, label="$\\theta$"
    )
    axes[0, 2].axhline(0, color="g", linestyle="--", alpha=0.5, label="Upright")
    axes[0, 2].set_ylabel("$\\theta$ (deg)", fontsize=11)
    axes[0, 2].set_xlabel("Time (s)", fontsize=11)
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].legend()
    axes[0, 2].set_title("Angle from Vertical")

    # --- Subplot 4: Velocities ---
    axes[1, 0].plot(t_plot, X_traj[:, 3], "b-", linewidth=2, label="$v_x$")
    axes[1, 0].plot(t_plot, X_traj[:, 4], "r-", linewidth=2, label="$v_z$")
    axes[1, 0].axhline(0, color="k", linestyle="--", alpha=0.3)
    axes[1, 0].set_ylabel("Velocity (m/s)", fontsize=11)
    axes[1, 0].set_xlabel("Time (s)", fontsize=11)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    axes[1, 0].set_title("Velocities")

    # --- Subplot 5: Control Inputs ---
    axes[1, 1].plot(t_control, U_traj[:, 0], "b-", linewidth=2, label="$T_x$")
    axes[1, 1].plot(t_control, U_traj[:, 1], "r-", linewidth=2, label="$T_z$")
    if hover_thrust is not None:
        axes[1, 1].axhline(
            hover_thrust, color="g", linestyle="--", alpha=0.5, label="Hover"
        )
    if u_min is not None and u_max is not None:
        axes[1, 1].axhline(u_min[0], color="gray", linestyle=":", alpha=0.5)
        axes[1, 1].axhline(u_max[0], color="gray", linestyle=":", alpha=0.5)
    axes[1, 1].set_ylabel("Thrust (N)", fontsize=11)
    axes[1, 1].set_xlabel("Time (s)", fontsize=11)
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    axes[1, 1].set_title("Control Inputs")

    # --- Subplot 6: 2D Trajectory ---
    x_range = max(3, np.max(np.abs(X_traj[:, 0])) + 1)

    # Draw environment if obstacles provided
    draw_environment(
        axes[1, 2],
        landing_pad_radius,
        x_range,
        water_level=0,
        obstacle1=obstacle1,
        obstacle2=obstacle2,
        rocket_radius=rocket_radius,
    )
    # Trajectory
    axes[1, 2].plot(
        X_traj[:, 0], X_traj[:, 1], "b-", linewidth=2, alpha=0.7, label="Path"
    )

    # Start marker
    axes[1, 2].plot(
        X_traj[0, 0], X_traj[0, 1], "go", markersize=12, label="Start", zorder=10
    )

    # Final rocket with velocity fire
    final_state = X_traj[-1]
    p_x_f, p_z_f, theta_f, v_x_f, v_z_f, omega_f = final_state
    draw_velocity_fire(axes[1, 2], p_x_f, p_z_f, theta_f, v_x_f, v_z_f, zorder=8)
    draw_rocket(axes[1, 2], p_x_f, p_z_f, theta_f, zorder=10)

    # Setup axes
    axes[1, 2].set_xlabel("$p_x$ (m)", fontsize=11)
    axes[1, 2].set_ylabel("$p_z$ (m)", fontsize=11)
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].legend(fontsize=9, loc="upper right")
    axes[1, 2].set_title("2D Trajectory")
    axes[1, 2].set_xlim(-x_range, x_range)
    axes[1, 2].set_ylim(-0.5, initial_state[1] + 0.5)
    axes[1, 2].set_aspect("equal")

    plt.tight_layout()
    plt.show()


def create_interactive_iteration_viewer(
    trajectories,
    initial_state,
    obstacle1,
    obstacle2,
    rocket_radius,
    landing_pad_radius,
    title_prefix="SQP Iteration",
):
    """
    Create interactive widget to view trajectory evolution over iterations.

    Args:
        trajectories: list of (N+1, 6) trajectory arrays
        initial_state: (6,) initial state
        obstacle1, obstacle2: obstacle dicts
        rocket_radius: safety buffer
        landing_pad_radius: landing zone radius
        title_prefix: prefix for plot titles
    """

    def plot_iteration(iteration=0):
        """Plot trajectory at specific iteration"""
        fig, ax = plt.subplots(figsize=(12, 12))

        # Get trajectory at this iteration
        X_iter = trajectories[iteration]
        final_state = X_iter[-1]
        p_x, p_z, theta, v_x, v_z, omega = final_state

        # Draw environment
        x_range = max(4, np.max(np.abs(X_iter[:, 0])) + 1)
        draw_environment(
            ax,
            obstacle1,
            obstacle2,
            rocket_radius,
            landing_pad_radius,
            x_range,
            water_level=0,
        )

        # Draw trajectory
        ax.plot(
            X_iter[:, 0],
            X_iter[:, 1],
            "b-",
            linewidth=3,
            alpha=0.8,
            label=f"Iteration {iteration}",
        )

        # Start marker
        ax.plot(
            X_iter[0, 0], X_iter[0, 1], "go", markersize=12, label="Start", zorder=10
        )

        # Draw rocket with velocity fire at final position
        draw_velocity_fire(ax, p_x, p_z, theta, v_x, v_z, zorder=8)
        draw_rocket(ax, p_x, p_z, theta, zorder=10)

        # Setup axes
        ax.set_xlabel("$p_x$ (m)", fontsize=14)
        ax.set_ylabel("$p_z$ (m)", fontsize=14)
        ax.set_title(
            f"{title_prefix} {iteration}/{len(trajectories) - 1}",
            fontsize=16,
            fontweight="bold",
        )
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=11, loc="upper right")
        ax.set_xlim(-x_range, x_range)
        ax.set_ylim(-1.0, initial_state[1] + 1.0)
        ax.set_aspect("equal")

        # Add iteration info
        final_pos = final_state[:2]
        final_vel = final_state[3:5]
        final_speed = np.linalg.norm(final_vel)
        final_distance = np.linalg.norm(final_pos)

        iter_info = (
            f"Iteration: {iteration}/{len(trajectories) - 1}\n"
            f"Final position: ({final_pos[0]:.2f}, {final_pos[1]:.2f}) m\n"
            f"Final angle: {theta * 180 / np.pi:.1f}°\n"
            f"Final velocity: ({v_x:.2f}, {v_z:.2f}) m/s\n"
            f"Final speed: {final_speed:.2f} m/s\n"
            f"Distance to origin: {final_distance:.2f} m"
        )

        ax.text(
            0.02,
            0.98,
            iter_info,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=12,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.9),
        )

        plt.tight_layout()
        plt.show()

    # Create slider
    slider = IntSlider(
        value=0,
        min=0,
        max=len(trajectories) - 1,
        step=1,
        description="Iteration:",
        continuous_update=False,
        layout=widgets.Layout(width="70%"),
    )

    # Create play button
    play = Play(
        value=0,
        min=0,
        max=len(trajectories) - 1,
        step=1,
        interval=500,
        description="Press play",
        disabled=False,
    )

    # Link play to slider
    jslink((play, "value"), (slider, "value"))

    # Create output
    out = interactive_output(plot_iteration, {"iteration": slider})

    display(HBox([play, slider]))
    display(out)


def plot_circle(ax, center: jnp.ndarray, radius: jnp.ndarray):
    """
    Plot obstacle locations as small circles.
    """
    px, py = center
    radius = radius[0]
    circle = Circle((px, py), radius, color="orange", alpha=0.6, zorder=5)
    ax.add_patch(circle)
    circle.set_label("Obstacle")


def simulate_open_loop(
    initial_state: np.ndarray,
    controls: np.ndarray,
    dynamics_func,
    dt: float,
) -> np.ndarray:
    """
    Simulate trajectory given initial state and control sequence.

    Args:
        initial_state: (n,) array of initial state
        controls: (N, m) array of control inputs
        dynamics_func: Function(state, control, params) -> next_state
        params: Additional parameters for dynamics function
        dt: Time step
    Returns:
        trajectory: (N+1, n) array of states over time
    """

    def _scan_func(carry, u):
        state = carry
        next_state = dynamics_func(state, u, dt)
        return next_state, next_state

    _, trajectory = jax.lax.scan(_scan_func, initial_state, controls)
    trajectory = jnp.vstack([initial_state, trajectory])
    return trajectory


def simulate_closed_loop(
    initial_state: np.ndarray,
    policy,
    n_steps: int,
    dynamics_func,
    dt: float,
    has_numpy=False,
) -> np.ndarray:
    """
    Simulate trajectory given initial state and control sequence.

    Args:
        initial_state: (n,) array of initial state
        policy: Function(state, time) -> control
        n_steps: Number of time steps to simulate
        dynamics_func: Function(state, control, dt) -> next_state
        dt: Time step
    Returns:
        trajectory: (N+1, n) array of states over time
    """

    def _scan_func(carry, _):
        state, time = carry
        control = policy(state, time)
        next_state = dynamics_func(state, control, dt)
        return (next_state, time + dt), next_state

    def scan(f, init, xs, length=None):
        if xs is None:
            xs = [None] * length
        carry = init
        ys = []
        for x in xs:
            carry, y = f(carry, x)
            ys.append(y)
        return carry, np.stack(ys)

    if has_numpy:
        _, trajectory = scan(_scan_func, (initial_state, 0), None, length=n_steps)
    else:
        _, trajectory = jax.lax.scan(
            _scan_func, (initial_state, 0), None, length=n_steps
        )
    trajectory = jnp.vstack([initial_state, trajectory])
    return trajectory


In [ ]:
# @title Visualize free body diagram of the rocket
fig, ax = plt.subplots(figsize=(8, 8))

ax.set_xlim(-2, 3)
ax.set_ylim(-0.5, 4)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Free Body Diagram", fontsize=16, fontweight="bold", pad=20)

# Rocket configuration
base_x, base_z = 0.5, 1.5
base_w, base_h = 0.6, 0.3
theta_example = 25 * np.pi / 180  # 25 degree tilt
ell = 2.0

# Rocket base (cart)
base = patches.Rectangle(
    (base_x - base_w / 2, base_z - base_h / 2),
    base_w,
    base_h,
    facecolor="gray",
    alpha=0.6,
    linewidth=2,
    edgecolor="black",
)
ax.add_patch(base)

# Rocket body (pole)
body_top_x = base_x + ell * np.sin(theta_example)
body_top_z = base_z + ell * np.cos(theta_example)
ax.plot(
    [base_x, body_top_x],
    [base_z, body_top_z],
    "b-",
    linewidth=10,
    solid_capstyle="round",
    alpha=0.6,
)

# Center of mass of body
cm_x = base_x + 0.5 * ell * np.sin(theta_example)
cm_z = base_z + 0.5 * ell * np.cos(theta_example)
ax.plot(cm_x, cm_z, "ro", markersize=12, label="Body CM", zorder=5)

# Angle theta annotation
arc_radius = 0.5
arc = patches.Arc(
    (base_x, base_z),
    2 * arc_radius,
    2 * arc_radius,
    angle=0,
    theta1=90 - theta_example * 180 / np.pi,
    theta2=90,
    color="darkred",
    linewidth=2,
)
ax.add_patch(arc)
ax.text(
    base_x + 0.05,
    base_z + 0.55,
    r"$\theta$",
    fontsize=14,
    color="darkred",
    fontweight="bold",
)

# Vertical reference line
ax.plot(
    [base_x, base_x],
    [base_z, base_z + 2.2],
    "k--",
    linewidth=1.5,
    alpha=0.4,
    label="Vertical",
)

mid_x = base_x + 0.5 * ell * np.sin(theta_example)
mid_z = base_z + 0.5 * ell * np.cos(theta_example)
ax.plot(
    [mid_x - 0.15 * np.cos(theta_example), mid_x + 0.15 * np.cos(theta_example)],
    [mid_z + 0.15 * np.sin(theta_example), mid_z - 0.15 * np.sin(theta_example)],
    "k-",
    linewidth=2,
)
ax.text(
    mid_x - 0.25 * np.cos(theta_example),
    mid_z + 0.25 * np.sin(theta_example),
    r"$\frac{\ell}{2}$",
    fontsize=13,
    fontweight="bold",
)

# Forces on base
thrust_scale = 0.9
T_x_mag, T_z_mag = 0.8, 1.3

# Horizontal thrust
ax.arrow(
    base_x,
    base_z,
    T_x_mag * thrust_scale,
    0,
    head_width=0.18,
    head_length=0.15,
    fc="red",
    ec="red",
    linewidth=3,
)
ax.text(
    base_x + T_x_mag * thrust_scale + 0.2,
    base_z,
    "$T_x$",
    fontsize=14,
    color="red",
    fontweight="bold",
    va="center",
)

# Vertical thrust
ax.arrow(
    base_x,
    base_z,
    0,
    T_z_mag * thrust_scale,
    head_width=0.18,
    head_length=0.15,
    fc="orange",
    ec="orange",
    linewidth=3,
)
ax.text(
    base_x + 0.0,
    base_z + T_z_mag * thrust_scale + 0.2,
    "$T_z$",
    fontsize=14,
    color="orange",
    fontweight="bold",
)

# Gravity on base
ax.arrow(
    base_x,
    base_z,
    0,
    -0.8,
    head_width=0.18,
    head_length=0.15,
    fc="blue",
    ec="blue",
    linewidth=3,
)
ax.text(
    base_x - 0.3, base_z - 0.9, "$m_t g$", fontsize=13, color="blue", fontweight="bold"
)

# Gravity on body
ax.arrow(
    cm_x,
    cm_z,
    0,
    -0.8,
    head_width=0.18,
    head_length=0.15,
    fc="purple",
    ec="purple",
    linewidth=3,
)
ax.text(
    cm_x + 0.1, cm_z - 0.5, "$m_b g$", fontsize=13, color="purple", fontweight="bold"
)

# Position labels
ax.text(
    base_x - 0.15,
    base_z - 0.05,
    "$(p_x, p_z)$",
    fontsize=12,
    ha="right",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

# Mass labels
ax.text(
    base_x,
    base_z - 0.5,
    "$m_t$",
    ha="center",
    fontsize=12,
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
)
ax.text(
    body_top_x + 0.1,
    body_top_z + 0.1,
    "$m_b, \ell$",
    ha="left",
    fontsize=12,
    bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8),
)

# Coordinate axes at lower left
arrow_len = 0.4
ax_x, ax_z = -1.5, 0.3
ax.annotate(
    "",
    xy=(ax_x + arrow_len, ax_z),
    xytext=(ax_x, ax_z),
    arrowprops=dict(arrowstyle="->", lw=2, color="black"),
)
ax.annotate(
    "",
    xy=(ax_x, ax_z + arrow_len),
    xytext=(ax_x, ax_z),
    arrowprops=dict(arrowstyle="->", lw=2, color="black"),
)
ax.text(ax_x + arrow_len + 0.1, ax_z - 0.1, "$x$", fontsize=13, fontweight="bold")
ax.text(ax_x + 0.1, ax_z + arrow_len + 0.05, "$z$", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## Derivation of equations of motion

Given the free body diagram, below are more details on deriving the equations of motion. Take a closer look at the derication if you are interested.
We will use the equations of motion to define the dynamics of our rocket.

The system dynamics are derived using the Lagrangian $L= T - V$.

Kinetic Energy
$$
T = \frac{1}{2}(m_t + m_b)(\dot{p}_x^2 + \dot{p}_z^2) + \frac{1}{2}m_b\ell\dot{\theta}(\dot{p}_x\cos\theta - \dot{p}_z\sin\theta) + \frac{1}{6}m_b\ell^2\dot{\theta}^2
$$

Potential Energy
$$
V = (m_t + m_b)gp_z + \frac{1}{2}m_bg\ell\cos\theta
$$


**modeling assumption:** The horizontal thrust $T_x$ is applied at the center of mass of the rocket body (height $\ell/2$ above the base), not at the base itself. This creates both:
1. A horizontal force on the system
2. A torque about the base: $\tau = \frac{\ell}{2} T_x$

This **thrust vectoring** capability is essential for controllability. In real rockets, engines are gimbaled to provide moment control. Without this, horizontal thrust at the base would have very weak coupling to the angle dynamics, making stabilization nearly impossible.

**Equations of Motion (Matrix Form)**: Applying Euler-Lagrange equations yields:

$$ \frac{d}{dt}\left( \frac{\partial L}{\partial \dot{q}}\right) - \frac{\partial L}{\partial q} = \sum_i Q_i $$
$$
\mathbf{M}(\theta)\ddot{q} + \mathbf{C}(\theta,\dot{\theta})\dot{q} + \mathbf{g}(\theta) = \mathbf{B}u
$$

where $q = [p_x, p_z, \theta]^\top$ and:

$$
\mathbf{M}(\theta) = \begin{bmatrix}
m_t + m_b & 0 & \frac{1}{2}m_b\ell\cos\theta \\
0 & m_t + m_b & -\frac{1}{2}m_b\ell\sin\theta \\
\frac{1}{2}m_b\ell\cos\theta & -\frac{1}{2}m_b\ell\sin\theta & \frac{1}{3}m_b\ell^2
\end{bmatrix}
$$

$$
\mathbf{C}(\theta, \dot{\theta}) = \begin{bmatrix}
0 & 0 & -\frac{1}{2}m_b\ell\dot{\theta}\sin\theta \\
0 & 0 & -\frac{1}{2}m_b\ell\dot{\theta}\cos\theta \\
0 & 0 & 0
\end{bmatrix}, \quad
\mathbf{g}(\theta) = \begin{bmatrix}
0 \\
(m_t + m_b)g \\
-\frac{1}{2}m_bg\ell\sin\theta
\end{bmatrix}
$$

$$
\mathbf{B}u = \begin{bmatrix}
T_x \\
T_z \\
\frac{\ell}{2}T_x
\end{bmatrix} \quad \text{(thrust vectoring)}
$$

### State-Space Form

The continuous dynamics are:
$$
\dot{x} = f(x, u) = \begin{bmatrix}
v_x \\ v_z \\ \omega \\
\mathbf{M}(\theta)^{-1}\left[\mathbf{B}u - \mathbf{C}(\theta,\omega)\begin{bmatrix}v_x \\ v_z \\ \omega\end{bmatrix} - \mathbf{g}(\theta)\right]
\end{bmatrix}
$$

We discretize using forward Euler: $x_{k+1} = x_k + \Delta t \cdot f(x_k, u_k)$


### (a) Implement the system dynamics (student response required)

In [ ]:
# [student response here]
# implement the rocket ode dynamics
def rocket_ode(states, controls, params):
    """
    Continuous-time dynamics for inverted pendulum rocket WITH THRUST VECTORING.

    The horizontal thrust T_x is applied at base, creating both:
    - Horizontal force on the base
    - Torque about the base = (l/2) x T_x

    Args:
        states: [p_x, p_z, theta, v_x, v_z, omega] - state vector
        controls: [T_x, T_z] - thrust forces
        params: [m_t, m_b, ell, g] - system parameters

    Returns:
        state derivatives: [v_x, v_z, omega, a_x, a_z, alpha]
    """
    m_t, m_b, ell, g = params
    mass = m_t + m_b
    p_x, p_z, theta, v_x, v_z, omega = states
    velocities = jnp.array([v_x, v_z, omega])
    T_x, T_z = controls

    # First derivatives (kinematic equations)
    first_derivs = jnp.array([v_x, v_z, omega])

    #### OUR CODE HERE ####

    # TODO: Compute Mass matrix M(theta)
    M = jnp.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
        ]
    ) # UPDATE ME

    # TODO: Compute Coriolis matrix C(theta, omega)
    C = jnp.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
        ]
    ) # UPDATE ME

    # TODO: Compute gravity vector g(theta)
    g_vec = jnp.array([0.0, 0.0, 0.0]) # UPDATE ME

    # TODO: Compute control input Bu with thrust vectoring
    # Hint: T_x creates both horizontal force and torque: τ = (l/2) x T_x
    # Bu should be [T_x, T_z, torque_from_T_x]
    Bu = jnp.array([0.0, 0.0, 0.0]) # UPDATE ME

    # TODO: Solve for accelerations using M * q_ddot = Bu - C * q_dot - g
    # Hint: Use jnp.linalg.solve
    accelerations = jnp.zeros(3) # UPDATE ME

    ####################################

    return jnp.concatenate([first_derivs, accelerations])


def runge_kutta_integrate(
    dynamics: Callable[[jnp.ndarray, jnp.ndarray, float], jnp.ndarray], dt: float
) -> Callable[[jnp.ndarray, jnp.ndarray, float], jnp.ndarray]:
    """
    Implement Runge-Kutta integration for discrete-time dynamics.

    Args:
        dynamics: A callable representing the continuous-time dynamics function.
        dt: The time step for integration.

    Returns:
        A callable representing the discrete-time dynamics using Runge-Kutta integration.
    """

    # zero-order hold
    def integrator(x: jnp.ndarray, u: jnp.ndarray, t: float) -> jnp.ndarray:
        # TODO: Implement Runge-Kutta integration here
        # raise NotImplementedError # remove this
        dt2 = dt / 2.0
        k1 = dynamics(x, u, t)
        k2 = dynamics(x + dt2 * k1, u, t + dt2)
        k3 = dynamics(x + dt2 * k2, u, t + dt2)
        k4 = dynamics(x + dt * k3, u, t + dt)
        return x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    return integrator


def rocket_dynamics_dt(state, controls, params, timestep):
    """
    Discrete-time dynamics using forward Euler integration.
        x_{k+1} = x_k + dt * f(x_k, u_k)

    Args:
        state: [p_x, p_z, theta, v_x, v_z, omega] - current state
        controls: [T_x, T_z] - thrust forces
        params: [m_t, m_b, ell, g] - system parameters
        timestep: time step dt

    Returns:
        next state x_{k+1}
    """
    # TODO: Implement forward Euler integration
    # Hint: Use rocket_ode to compute state derivative
    return state


### (b) PID controller

Now let's implement a PID (Proportional-Derivative) controller to land the rocket. Landing requires simultaneously keep rocket upright, move toward landing pad, and reduce altitude gradually.

We use a cascaded PID controller with two loops:

Primary loop:
$$T_x^{\text{angle}} = -K_{p,\theta}(\theta - 0) - K_{d,\theta}\omega$$

This keeps the rocket upright by applying horizontal thrust proportional to the tilt angle and angular velocity.

Secondary loop:
$$T_x^{\text{pos}} = -K_{p,x}(p_x - 0) - K_{d,x}v_x$$

This guides the rocket toward the landing pad center. The combined horizontal thrust is:
$$T_x = T_x^{\text{angle}} + T_x^{\text{pos}}$$

Vertical Thrust Profile:
We use a simple time-based linear ramp:
$$T_z(t) = T_{z,\text{init}} \left(1 - \frac{t}{T_{\text{sim}}}\right) + T_{z,\text{final}} \frac{t}{T_{\text{sim}}}$$

This gradually reduces thrust from an initial value (to slow descent) to a final value (for soft touchdown).


Your task is to experiment with the PID gains and thrust profile values to achieve a good landing! The goal is to:
- Land near the origin ($|p_x| < 1$ m, $p_z \approx 0$ m)
- Stay upright ($|\theta| < 20$ deg)
- Have low final velocity ($|v| < 1.5$ m/s)

In [ ]:
# @title Set up system parameters. (Run cell as is.)
m_t = 1.0  # kg - thruster/base mass
m_b = 0.5  # kg - body mass
ell = 2.0  # m - rocket body length
g = 9.81  # m/s^2 - gravity
params = jnp.array([m_t, m_b, ell, g])

# Simulation parameters
dt = 0.01  # time step (seconds)
T_sim = 5.0  # simulation time (seconds)
n_steps = int(T_sim / dt)

# Initial state: [p_x, p_z, theta, v_x, v_z, omega]
initial_state = jnp.array(
    [
        0.0,  # p_x: start at origin
        3.0,  # p_z: start 3m high
        0.02,  # theta: barely tilted (~1 degree)
        0.2,  # v_x: moving slowly at 0.2 m/s
        -1.0,  # v_z: no initial vertical velocity
        0.0,  # omega: no initial rotation
    ]
)


hover_thrust = (m_t + m_b) * g

rocket_dynamics_dt_partial = functools.partial(rocket_dynamics_dt, params=params)

In [ ]:
# @title PID Controller Implementation. (Run cell as is.)
def pid_controller(state, t, gains, T_z_profile, T_sim):
    """PID controller for rocket landing with thrust vectoring"""
    p_x, p_z, theta, v_x, v_z, omega = state
    K_p_angle, K_d_angle, K_p_pos_x, K_d_pos_x = gains
    T_z_init, T_z_fin = T_z_profile

    # Target: land at origin upright
    target_x = 0.0
    target_theta = 0.0

    # Angle control (stabilization) - primary
    T_x_angle = -K_p_angle * (theta - target_theta) - K_d_angle * omega

    # Position control (guidance) - secondary
    T_x_position = -K_p_pos_x * (p_x - target_x) - K_d_pos_x * v_x

    # Combined horizontal thrust
    T_x = T_x_angle + T_x_position

    # Vertical thrust: linear ramp
    alpha = t / T_sim
    T_z = T_z_init * (1 - alpha) + T_z_fin * alpha

    return jnp.array([T_x, T_z])


In [ ]:
# @title [student response here]

# TODO: Try out some gains, and see how well your controller performs!
# It's okay that you don't get it perfectly. The point is that tuning the gains can be challenging.


# Angle stabilization gains (keep rocket upright)
K_p_angle = 50.0  # Proportional gain UPDATE ME!
K_d_angle = 20.0  # Derivative gain UPDATE ME!

# Horizontal position control (move toward target)
K_p_pos_x = 20  # Proportional gain UPDATE ME!
K_d_pos_x = 10  # Derivative gain UPDATE ME!

# Vertical thrust profile (simple time-based)
T_z_initial = 14.0  # Initial vertical thrust UPDATE ME!
T_z_final = 13.3  # Final vertical thrust UPDATE ME!


policy = functools.partial(
    pid_controller,
    gains=(K_p_angle, K_d_angle, K_p_pos_x, K_d_pos_x),
    T_z_profile=(T_z_initial, T_z_final),
    T_sim=T_sim,
)

In [ ]:
# @title Run control and visualize results. (Run cell as is.)
trajectory = simulate_closed_loop(
    initial_state,
    policy=policy,
    n_steps=n_steps,
    dynamics_func=rocket_dynamics_dt_partial,
    dt=dt,
)

controls = jnp.array(
    [
        policy(state, time)
        for state, time in zip(trajectory[:-1], np.arange(trajectory.shape[0] - 1) * dt)
    ]
)

plot_trajectory_analysis(
    trajectory,
    controls,
    T_sim,
    initial_state,
    landing_pad_radius=0.5,
    hover_thrust=hover_thrust,
    title_prefix=f"PID Controller (K_p={K_p_angle:.1f}, K_d={K_d_angle:.1f})",
)

In [ ]:
# @title Animate rocket landing trajectory with interactive timestep slider. (Run cell as is.)
@interact(timestep=(0, n_steps - 1, 10))
def plot_trajectory(timestep):
    """
    Plot rocket landing trajectories over SQP iterations.
    """

    fig, ax = plt.subplots(figsize=(8, 5))
    px, pz, theta, vx, vz, omega = trajectory.T
    plt.plot(px, pz, alpha=0.3, color="k")

    selected_Tx, selected_Tz = controls[timestep]
    draw_rocket(ax, px[timestep], pz[timestep], theta[timestep], zorder=10)
    draw_velocity_fire(
        ax,
        px[timestep],
        pz[timestep],
        theta[timestep],
        selected_Tx,
        selected_Tz,
        zorder=8,
    )
    plt.fill_between(
        jnp.linspace(-5, 5, 2), 0, -5, color="lightblue", alpha=0.5, label="Water"
    )
    ax.set_xlabel("Position X (m)")
    ax.set_ylabel("Position Z (m)")
    ax.set_title("Rocket Landing Trajectories with PD controller")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 4)
    plt.axis("equal")
    ax.grid()
    plt.show()

### (c) At this point, we might as well try solve this using SQP!

To build an SQP algorithm, we need to linearize the nonlinear dynamics around a reference trajectory. This creates a local approximation that we can use in a quadratic program.

Given a reference trajectory $\{x_k^{\text{ref}}, u_k^{\text{ref}}\}$, we linearize the discrete dynamics around this trajectory:
$$
x_{k+1} = f(x_k, u_k) \approx
f(x_k^{\text{ref}}, u_k^{\text{ref}}) +
A_k (x_k - x_k^{\text{ref}}) + B_k (u_k - u_k^{\text{ref}}),
$$
where the Jacobian matrices are:
$$
A_k = \frac{\partial f}{\partial x}\bigg|_{(x_k^{\text{ref}}, u_k^{\text{ref}})},\quad
B_k = \frac{\partial f}{\partial u}\bigg|_{(x_k^{\text{ref}}, u_k^{\text{ref}})}
$$

We can rewrite this as an **affine model**:
$$
x_{k+1} \approx A_k x_k + B_k u_k + c_k,
$$
where the offset $c_k$ is chosen so the approximation is exact at the reference:
$$
c_k = f(x_k^{\text{ref}}, u_k^{\text{ref}}) - A_k x_k^{\text{ref}} - B_k u_k^{\text{ref}}
$$

In [ ]:
# @title First, we linearize the dynamics. (Run cell as is.)

# this is something you have seen in previous exercises
@eqx.filter_jit
def linearize_dynamics(rocket_dynamics_dt, state, control, dt):
    """
    Linearize the discrete-time dynamics around the given state.

    Args:
        rocket_dynamics_dt: Function(state, control, timestep) -> next_state
        state: (n,) array of state around which to linearize
        control: (m,) array of control around which to linearize
        dt: time step

    Returns:
        A: State transition matrix (n, n)
        B: Control input matrix (n, m)
    """

    A, B = jax.jacobian(rocket_dynamics_dt, argnums=(0, 1))(state, control, dt)
    C = rocket_dynamics_dt(state, control, dt) - A @ state - B @ control
    return A, B, C


### (c)(i) Set up the CVXPY problem (Student response required.)
In the provided code below, add comments for each line, and write down *using LaTeX* the trajectory optimization it is solving. Be as precise and accurate as you with the code. For example, use $\epsilon_k$ for referring to each element of the variable `epsilons` which is referring to a vector of $\epsilon$ values.

How many decision variables and constraints does this problem have?

[student response here]

In [ ]:
# [student response here]
# TODO: Comment this code block: This code sets up the variables and parameters for the convex optimization problem
# that will be solved using CVXPY to find an optimal control sequence for the rocket landing problem.
n_states = 6
m_controls = 2
n_steps = 275

u_min = np.array([-20.0, 0.0])
u_max = np.array([20.0, 25.0])

goal_state_lower = np.array([-1.5, 0.0, -0.1, -0.1, -0.1, -0.1])
goal_state_upper = np.array([1.5, 0.1, 0.1, 0.1, 0.0, 0.1])
slack_penalty = 3000.0

states = cp.Variable((n_steps + 1, n_states))
controls = cp.Variable((n_steps, m_controls))
epsilons = cp.Variable((n_steps + 1))

As = cp.Parameter((n_steps, n_states, n_states), name="As")
Bs = cp.Parameter((n_steps, n_states, m_controls), name="Bs")
Cs = cp.Parameter((n_steps, n_states), name="Cs")
previous_states_param = cp.Parameter((n_steps + 1, n_states), name="previous_states")
previous_controls_param = cp.Parameter((n_steps, m_controls), name="previous_controls")
Gs = cp.Parameter((n_steps + 1, n_states), name="Gs")
hs = cp.Parameter((n_steps + 1,), name="hs")
initial_state_param = cp.Parameter(n_states, name="initial_state")

Q = cp.Constant(np.diag([10.0, 5.0, 20.0, 10.0, 10.0, 15.0]))
R = cp.Constant(np.diag([0.1, 0.1]))
Qf = cp.Constant(np.diag([200.0, 1000.0, 100.0, 500.0, 1000.0, 100.0]))

trust_region_penalty = cp.Constant(100.0)

cost = slack_penalty * cp.sum(epsilons**2)
constraints = [epsilons >= 0]
constraints.append(states[0, :] == initial_state_param)
constraints.append(states[n_steps, :] <= goal_state_upper)
constraints.append(states[n_steps, :] >= goal_state_lower)

for k in range(n_steps):
    constraints.append(
        states[k + 1, :] == As[k] @ states[k, :] + Bs[k] @ controls[k, :] + Cs[k]
    )

    constraints.append(controls[k, :] >= u_min)
    constraints.append(controls[k, :] <= u_max)
    constraints.append(Gs[k, :] @ states[k, :] + hs[k] <= epsilons[k])

    cost += cp.quad_form(states[k, :], Q) + cp.quad_form(controls[k, :], R)
    cost += trust_region_penalty * cp.sum_squares(
        states[k, :] - previous_states_param[k, :]
    ) + trust_region_penalty * cp.sum_squares(
        controls[k, :] - previous_controls_param[k, :]
    )

cost += cp.quad_form(states[n_steps, :], Qf)
cost += trust_region_penalty * cp.sum_squares(
    states[n_steps, :] - previous_states_param[n_steps, :]
)

problem = cp.Problem(cp.Minimize(cost), constraints)


### Some additional functions and code to set up the problem. (Run cell as it)

In [ ]:
# set up obstacle constraint function
def _obstacle_constraint(state, obs_center, obs_radius):
    """Compute obstacle constraint value"""
    p_x, p_z = state[0], state[1]
    obs_x, obs_z = obs_center
    dist_sq = (p_x - obs_x) ** 2 + (p_z - obs_z) ** 2
    return obs_radius[0] ** 2 - dist_sq

# obstacle parameters
obs_center = jnp.array([-0.3, 2.0])
obs_radius = jnp.array([0.4])

obstacle_constraint = functools.partial(
    _obstacle_constraint, obs_center=obs_center, obs_radius=obs_radius
)

In [ ]:
# set initial state
initial_state_param.value = np.array(
    [
        0.0,  # p_x: start at origin
        3.0,  # p_z: start 3m high
        0.01,  # theta: barely tilted (~1 degree)
        0.2,  # v_x: moving slowly at 0.2 m/s
        -0.5,  # v_z: small initial vertical velocity
        0.01,  # omega: small initial rotation
    ]
)


### (c)(ii) Now, we set up the single SQP problem and the also loop that will run the multiple SQP iterations.
In the cell block below, add comments to the code to explain what each step is doing.

In [ ]:
# [student response here]
# TODO: comment the two functions below: These functions implement the Sequential Quadratic Programming (SQP)
# algorithm to iteratively solve the trajectory optimization problem for the rocket landing scenario.

def sqp_solve(
    problem: cp.Problem,
    previous_states: np.ndarray,
    previous_controls: np.ndarray,
    solver=cp.CLARABEL,
    verbose=False,
):
    """
    Solve the SQP subproblem given previous trajectory.

    Args:
        problem: CVXPY problem instance
        previous_states: (N+1, n) array of previous states
        previous_controls: (N, m) array of previous controls
        solver: CVXPY solver to use
        verbose: Whether to print solver output

    Returns:
        new_states: (N+1, n) array of optimized states
        new_controls: (N, m) array of optimized controls
    """

    As_jnp, Bs_jnp, Cs_jnp = jax.vmap(linearize_dynamics, in_axes=(None, 0, 0, None))(
        rocket_dynamics_dt_partial, previous_states[:-1], previous_controls, dt
    )

    Gs_jnp = jax.vmap(jax.jacobian(obstacle_constraint, argnums=0), in_axes=(0,))(
        previous_states
    )
    hs_jnp = jax.vmap(obstacle_constraint)(previous_states) - jax.vmap(
        jnp.dot, in_axes=(0, 0)
    )(Gs_jnp, previous_states)

    As.value = np.array(As_jnp)
    Bs.value = np.array(Bs_jnp)
    Cs.value = np.array(Cs_jnp)
    Gs.value = np.array(Gs_jnp)
    hs.value = np.array(hs_jnp)

    previous_states_param.value = np.array(previous_states)
    previous_controls_param.value = np.array(previous_controls)

    problem.solve(solver=solver, warm_start=True, ignore_dpp=True, verbose=verbose)

    new_states = states.value
    new_controls = controls.value

    return new_states, new_controls


def run_sqp_loop(
    problem, initial_states, initial_controls, n_iterations=10, solver=cp.OSQP, verbose=False
):
    """
    Run SQP loop to optimize trajectory.

    Args:
        problem: CVXPY problem instance
        initial_states: (N+1, n) array of initial states
        initial_controls: (N, m) array of initial controls
        n_iterations: Number of SQP iterations
        solver: CVXPY solver to use
        verbose: Whether to print solver output

    Returns:
        states: List of state states over iterations
        controls: List of control states over iterations
    """
    states = [initial_states]
    controls = [initial_controls]

    current_states = initial_states
    current_controls = initial_controls

    for iteration in range(n_iterations):
        print(f"SQP Iteration {iteration + 1}/{n_iterations}")
        new_states, new_controls = sqp_solve(
            problem, current_states, current_controls, solver=solver, verbose=verbose
        )
        states.append(new_states)
        controls.append(new_controls)

        current_states = new_states
        current_controls = new_controls

    return states, controls

### (c)(iii) Run the SQP loop!
Now, we will run the SQP and visualize the results.
In the visualization, you should see the solutions from each SQP iteration.
Take a closer look at the results.

Try out some different parameters for the problem, such as trust region penalty, cost matrices, slack penalty, initial state etc.
Comment on your observations, such as describing (briefly) the influence of certain parameters on the solution.

[student response here]

In [ ]:
# @title Use PID control results as nominal trajectory for linearization. (Run cell as is.)

trajectory_nominal = simulate_closed_loop(
    initial_state,
    policy=policy,
    n_steps=n_steps,
    dynamics_func=rocket_dynamics_dt_partial,
    dt=dt,
)

controls_nominal = jnp.array(
    [
        policy(state, time)
        for state, time in zip(trajectory_nominal[:-1], np.arange(n_steps) * dt)
    ]
)

# test run of sqp_solve, will "compile" problem to make future solves faster
# note, a warning may be shown about the solution being inaccurate.
# This is expected on the first solve due to the initial guess being poor.
# and that the problem has an terminal state constraint which is hard to satisfy given the fixed horizon length.
# This is one challenge of using SQP for trajectory optimization with a fixed time horizon.
# It is possible to frame the problem with free final time to help alleviate this issue.
# But the formulation is more complex and beyond the scope of this exercise.
verbose = True # set to True to see solver output, otherwise set to False to reduce output
new_states, new_controls = sqp_solve(
    problem, trajectory_nominal, controls_nominal, solver=cp.CLARABEL, verbose=verbose
)


In [ ]:
# @title Run full SQP loop to optimize trajectory. (Run cell as is.)
# Note: Depending on your machine, this may take around one minute or so to run.
n_iterations = 20 # feel free to adjust number of iterations
states_list, controls_list = run_sqp_loop(
    problem, trajectory_nominal, controls_nominal, n_iterations=n_iterations, solver=cp.CLARABEL
)

In [ ]:
# @title Visualize results. (Run cell as is.)
@interact(iteration=(0, len(states_list) - 1), timestep=(0, n_steps - 1, 10))
def plot_sqp_trajectories(iteration, timestep):
    """
    Plot rocket landing trajectories over SQP iterations.
    """

    fig, ax = plt.subplots(figsize=(8, 5))
    for trajectory in states_list:
        px, pz, theta, vx, vz, omega = trajectory.T
        plt.plot(px, pz, alpha=0.3, color="k")
    ax.plot(states_list[0][:, 0], states_list[0][:, 1], "r--", label="Initial guess")
    ax.plot(
        states_list[-1][:, 0],
        states_list[-1][:, 1],
        "g-",
        linewidth=2,
        label="Final optimized",
    )
    ax.plot(
        states_list[iteration][:, 0],
        states_list[iteration][:, 1],
        "b-",
        linewidth=2,
        label="Selected iteration",
    )
    selected_px, selected_pz, selected_theta, v_x, v_z, omega = states_list[iteration][
        timestep
    ]
    selected_Tx, selected_Tz = controls_list[iteration][timestep]
    final_px, final_pz, final_theta, v_x, v_z, omega = states_list[iteration][-1]
    draw_rocket(ax, selected_px, selected_pz, selected_theta, zorder=10)
    draw_velocity_fire(
        ax, selected_px, selected_pz, selected_theta, selected_Tx, selected_Tz, zorder=8
    )
    plot_circle(ax, obs_center, obs_radius)
    ax.set_xlabel("Position X (m)")
    ax.set_ylabel("Position Z (m)")
    ax.set_title(
        "Rocket Landing Trajectories over SQP Iterations. \n Final state: ({:.2f}, {:.2f}, {:.2f}, {:.2f}, {:.2f}, {:.2f})".format(
            final_px, final_pz, final_theta, v_x, v_z, omega
        )
    )
    final_state = states_list[iteration][-1]
    eps = 1E-3 # some tolerance on checking final state constraint
    final_state_constraint = (final_state <= goal_state_upper + eps).all() and (final_state >= goal_state_lower - eps).all()
    print(f"Final state within goal region: {final_state_constraint}")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-1, 6)
    plt.axis("equal")
    ax.legend()
    ax.grid()
    plt.show()